# 04 BookRAG（論文忠実版 / arXiv:2512.03413）

文書ネイティブの **BookIndex = (Tree T, KG G, GT-Link M)** を構築し、
Information Foraging Theory に基づく **エージェント型検索**（クエリ分類 → Selector → Reasoner → Skyline → Synthesizer）で回答する。

- **Tree**: 文書の論理階層（目次）。Section / Text / Table / Image
- **KG**: ノードから抽出したエンティティ・関係（Gradient-based Entity Resolution で名寄せ）
- **GT-Link**: エンティティ → ツリーノードの対応

> 取り込み時に LLM でエンティティ抽出・名寄せを行うため、PagedRAG より時間と API 消費が大きい。
> 構造の濃い長尺文書（ハンドブック・規程・論文）で効果が出る。

In [ ]:
import llmlab

llmlab.settings_form()  # Base URL / API Key / Model / Embed Model を入力

In [ ]:
book = llmlab.BookRAG()

# 文書を取り込み BookIndex を構築（Markdown は見出し階層をそのまま木にできる）
book.add_book("../docs/handbook.pdf", title="Handbook")

book.info()   # 木のノード数 / エンティティ数 / 関係数

## クエリ分類に応じてプランが切り替わる

- **single-hop**: Extract → Select_by_Entity/Section → (Graph∥Text) → Skyline → Reduce
- **multi-hop**: Decompose → 各サブ問題に single-hop → Map → Reduce
- **global**: Filter_Modal/Range → Map → Reduce

In [ ]:
ans = book.ask("How does X differ from Y?")
print(ans)            # 回答 + 分類/プラン + 根拠ノード（書名・ページ・スコア）

In [ ]:
# 構造化アクセス
print("分類:", ans.category)
print("プラン:", ans.plan)
for e in ans.evidence:
    print(e.title, e.page, round(e.s_graph, 3), round(e.s_text, 3))